In [0]:
import hashlib
import json
import requests
import io
from datetime import datetime, timezone
import pyspark.sql.functions as F

In [0]:
COMPANY_CIK = "CIK0001045810"

# `/submissions` Endpoint

## Setting up schemas and tables

In [0]:
%sql
USE CATALOG edgar;
CREATE SCHEMA IF NOT EXISTS stg;
CREATE SCHEMA IF NOT EXISTS fct;
CREATE SCHEMA IF NOT EXISTS mrt;
USE SCHEMA stg;
CREATE TABLE IF NOT EXISTS stg_submissions_raw;
CREATE TABLE IF NOT EXISTS edgar.stg.ingestion_control (
    cik STRING,
    endpoint STRING,
    payload_hash STRING,
    max_acceptance_datetime STRING,
    last_fetch_at TIMESTAMP
);
-- The control table is now keyed on (cik, endpoint) so both endpoints can track one CIK.
-- One-time migration of the existing table (add + backfill `endpoint`) runs in the next Python cell.
-- Thin snapshot table for /xbrl: the raw payload stays in the volume, not in the table.
CREATE TABLE IF NOT EXISTS edgar.stg.stg_xbrl_raw (
    cik STRING,
    entity_name STRING,
    max_filed STRING,
    volume_path STRING,
    ingested_at TIMESTAMP,
    endpoint STRING
);
CREATE VOLUME IF NOT EXISTS edgar.stg.vol;

In [0]:
# One-time migration: the control table predates the `endpoint` column (keyed on (cik, endpoint) now).
cols = {r["col_name"] for r in spark.sql("DESCRIBE TABLE edgar.stg.ingestion_control").collect()}
if "endpoint" not in cols:
    spark.sql("ALTER TABLE edgar.stg.ingestion_control ADD COLUMN endpoint STRING")
    print("Added `endpoint` column to edgar.stg.ingestion_control.")
spark.sql("UPDATE edgar.stg.ingestion_control SET endpoint = '/submissions' WHERE endpoint IS NULL")
print("Existing control rows backfilled with endpoint = '/submissions'.")

## Submissions Data

In [0]:
headers = {"User-Agent": "User Agent"}
nvidia = requests.get(f"https://data.sec.gov/submissions/{COMPANY_CIK}.json", headers=headers)
data = nvidia.json()

# Stable serialization + content hash for idempotency
payload_str = json.dumps(data, sort_keys=True)
payload_hash = hashlib.sha256(payload_str.encode()).hexdigest()
max_acc = max(data["filings"]["recent"]["acceptanceDateTime"])

# Look up the last ingested hash for this CIK + endpoint
cik = str(data["cik"])
ctrl_rows = (spark.table("edgar.stg.ingestion_control")
             .filter((F.col("cik") == cik) & (F.col("endpoint") == "/submissions"))
             .collect())
stored_hash = ctrl_rows[0]["payload_hash"] if ctrl_rows else None
changed = stored_hash != payload_hash
print(f"Hash {'differs from' if changed else 'matches'} stored hash — payload {'will be' if changed else 'will NOT be'} ingested.")


In [0]:
if changed:
    # (i) write raw JSON file to the volume
    dbutils.fs.put(f"/Volumes/edgar/stg/vol/raw_submissions/{COMPANY_CIK}.json", payload_str, overwrite=True)
    print("File uploaded successfully!")

    # (ii) append snapshot to stg_submissions_raw
    stg_df = spark.createDataFrame([{
        "cik": cik,
        "ticker": ",".join(data.get("tickers", [])),
        "company_name": data["name"],
        "raw_payload": payload_str,
        "ingested_at": datetime.now(timezone.utc),
        "endpoint": "/submissions"
    }])
    stg_df.write.mode("append").saveAsTable("edgar.stg.stg_submissions_raw", mergeSchema=True)

    # (iii) upsert the control row
    spark.sql(f"""
        MERGE INTO edgar.stg.ingestion_control AS t
        USING (SELECT '{cik}' AS cik, '/submissions' AS endpoint, '{payload_hash}' AS payload_hash,
                      '{max_acc}' AS max_acceptance_datetime,
                      current_timestamp() AS last_fetch_at) AS s
        ON t.cik = s.cik AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.payload_hash = s.payload_hash,
                                    t.max_acceptance_datetime = s.max_acceptance_datetime,
                                    t.last_fetch_at = s.last_fetch_at
        WHEN NOT MATCHED THEN INSERT (cik, endpoint, payload_hash, max_acceptance_datetime, last_fetch_at)
        VALUES (s.cik, s.endpoint, s.payload_hash, s.max_acceptance_datetime, s.last_fetch_at)
    """)
    print("New payload ingested: file written, stg row appended, control updated.")
else:
    print(f"No change since last fetch for CIK {cik} — skipping file write, stg append, and control update.")

# `/xbrl` Endpoint

In [0]:
headers = {"User-Agent": "User Agent"}
nvidia = requests.get(f"https://data.sec.gov/api/xbrl/companyfacts/{COMPANY_CIK}.json", headers=headers)
data = nvidia.json()

# Stable serialization + content hash for idempotency (same pattern as /submissions)
payload_str = json.dumps(data, sort_keys=True)
payload_hash = hashlib.sha256(payload_str.encode()).hexdigest()

# companyfacts has no top-level acceptanceDateTime; recency lives per fact.
# Compute max `filed` across every fact (taxonomy -> concept -> unit -> fact list) for observability.
all_facts = [
    fact
    for concepts in data["facts"].values()
    for concept in concepts.values()
    for fact_list in concept["units"].values()
    for fact in fact_list
]
max_filed = max(fact["filed"] for fact in all_facts)

# Look up the last ingested hash for this CIK + endpoint
# companyfacts returns cik as an int (no zero-padding) — pad to match the /submissions CIK format
cik = str(data["cik"]).zfill(10)
ctrl_rows = (spark.table("edgar.stg.ingestion_control")
             .filter((F.col("cik") == cik) & (F.col("endpoint") == "/xbrl"))
             .collect())
stored_hash = ctrl_rows[0]["payload_hash"] if ctrl_rows else None
changed = stored_hash != payload_hash
print(f"Hash {'differs from' if changed else 'matches'} stored hash — payload {'will be' if changed else 'will NOT be'} ingested.")
print(f"Max filed date across {len(all_facts)} facts: {max_filed}")

In [0]:
if changed:
    # (i) write raw JSON file to the volume — the full payload lives here, not in the table
    volume_path = f"/Volumes/edgar/stg/vol/raw_xbrl/{COMPANY_CIK}.json"
    dbutils.fs.put(volume_path, payload_str, overwrite=True)
    print("File uploaded successfully!")

    # (ii) append a THIN snapshot row to stg_xbrl_raw
    stg_df = spark.createDataFrame([{
        "cik": cik,
        "entity_name": data.get("entityName"),
        "max_filed": max_filed,
        "volume_path": volume_path,
        "ingested_at": datetime.now(timezone.utc),
        "endpoint": "/xbrl"
    }])
    stg_df.write.mode("append").saveAsTable("edgar.stg.stg_xbrl_raw", mergeSchema=True)

    # (iii) upsert the control row
    spark.sql(f"""
        MERGE INTO edgar.stg.ingestion_control AS t
        USING (SELECT '{cik}' AS cik, '/xbrl' AS endpoint, '{payload_hash}' AS payload_hash,
                      '{max_filed}' AS max_acceptance_datetime,
                      current_timestamp() AS last_fetch_at) AS s
        ON t.cik = s.cik AND t.endpoint = s.endpoint
        WHEN MATCHED THEN UPDATE SET t.payload_hash = s.payload_hash,
                                    t.max_acceptance_datetime = s.max_acceptance_datetime,
                                    t.last_fetch_at = s.last_fetch_at
        WHEN NOT MATCHED THEN INSERT (cik, endpoint, payload_hash, max_acceptance_datetime, last_fetch_at)
        VALUES (s.cik, s.endpoint, s.payload_hash, s.max_acceptance_datetime, s.last_fetch_at)
    """)
    print("New payload ingested: file written, thin stg row appended, control updated.")
else:
    print(f"No change since last fetch for CIK {cik} — skipping file write, stg append, and control update.")